# Uda-hub CultPass Multi-Agent Support Application Notebook

Demonstrates end-to-end ticket processing workflow across 4 distinct ticket scenarios with classification, routing, tool usage, resolution, escalation, and structured event logging.

## 1. Setup & Environment

In [1]:
from dotenv import load_dotenv
from agentic.workflow import orchestrator, classify_ticket
from agentic.logger import log_event, get_ticket_events, get_metrics_summary
from agentic.tools import search_knowledge_base, get_user_profile, get_user_reservations, update_ticket_status, escalate_ticket

load_dotenv()
print("✅ Environment initialized successfully.")

ModuleNotFoundError: No module named 'agentic'

## 2. Ticket Scenario 1: Policy / FAQ Query (Successful Resolution)

- **Query**: How to cancel or pause subscription?
- **Expected Action**: Classified as `policy` -> Routed to `support_agent` -> RAG search -> Resolved in DB.

In [1]:
ticket_id_1 = "demo_ticket_policy_001"
user_msg_1 = "How do I cancel or pause my CultPass subscription?"
meta_1 = {"tags": "cancellation, policy", "urgency": "normal", "ticket_id": ticket_id_1}

# 1. Classification & Routing
cls_1 = classify_ticket(user_msg_1, meta_1)
print(f"[CLASSIFICATION]: {cls_1}")
log_event(ticket_id_1, "CLASSIFICATION", agent_name="supervisor_agent", details=cls_1)
log_event(ticket_id_1, "ROUTING", agent_name="supervisor_agent", details={"destination": cls_1["selected_agent"]})

# 2. Tool Execution
kb_res_1 = search_knowledge_base.invoke({"query": user_msg_1, "top_k": 3})
print(f"[RAG RESULT]: Highest Confidence = {kb_res_1['highest_confidence']}, Escalation = {kb_res_1['should_escalate']}")
log_event(ticket_id_1, "TOOL_CALL", tool_name="search_knowledge_base", details={"query": user_msg_1})
log_event(ticket_id_1, "TOOL_RESULT", tool_name="search_knowledge_base", outcome="success", details={"matched_articles": len(kb_res_1['articles'])})

# 3. Resolution & Database Update
upd_1 = update_ticket_status.invoke({"ticket_id": ticket_id_1, "status": "resolved", "issue_type": "cancellation"})
log_event(ticket_id_1, "RESOLUTION", agent_name="support_agent", details=upd_1)
print(f"[FINAL STATUS]: {upd_1}")

NameError: name 'classify_ticket' is not defined

## 3. Ticket Scenario 2: Low-Confidence Knowledge -> Human Escalation

- **Query**: Inquiry regarding non-existent feature.
- **Expected Action**: Classified -> RAG search returns `should_escalate: True` -> Escalated in DB.

In [1]:
ticket_id_2 = "demo_ticket_escalate_002"
user_msg_2 = "I need help with quantum portal integration feature 999."
meta_2 = {"tags": "unknown, bug", "urgency": "high", "ticket_id": ticket_id_2}

cls_2 = classify_ticket(user_msg_2, meta_2)
print(f"[CLASSIFICATION]: {cls_2}")
log_event(ticket_id_2, "CLASSIFICATION", agent_name="supervisor_agent", details=cls_2)

kb_res_2 = search_knowledge_base.invoke({"query": user_msg_2, "top_k": 3})
print(f"[RAG CONFIDENCE]: {kb_res_2['highest_confidence']}, Should Escalate: {kb_res_2['should_escalate']}")
log_event(ticket_id_2, "TOOL_CALL", tool_name="search_knowledge_base", details={"query": user_msg_2})

if kb_res_2["should_escalate"]:
    esc_res = escalate_ticket.invoke({"ticket_id": ticket_id_2, "reason": kb_res_2["escalation_reason"]})
    log_event(ticket_id_2, "ESCALATION", agent_name="support_agent", tool_name="escalate_ticket", outcome="escalated", details=esc_res)
    print(f"[ESCALATION RESULT]: {esc_res}")

NameError: name 'classify_ticket' is not defined

## 4. Ticket Scenario 3: Account & Reservation Services

- **Query**: Look up user profile and active reservations for `a4ab87`.
- **Expected Action**: Classified as `account` -> Routed to `account_agent` -> Profile & Reservations fetched -> Resolved.

In [1]:
ticket_id_3 = "demo_ticket_account_003"
user_msg_3 = "Please check remaining quota and reservations for user a4ab87."
meta_3 = {"tags": "profile, quota", "urgency": "normal", "ticket_id": ticket_id_3}

cls_3 = classify_ticket(user_msg_3, meta_3)
print(f"[CLASSIFICATION]: {cls_3}")
log_event(ticket_id_3, "CLASSIFICATION", agent_name="supervisor_agent", details=cls_3)

prof = get_user_profile.invoke({"user_id_or_email": "a4ab87"})
resv = get_user_reservations.invoke({"user_id": "a4ab87"})
print(f"[USER PROFILE]: Name = {prof.get('full_name')}, Tier = {prof.get('subscription', {}).get('tier')}")
print(f"[RESERVATIONS]: Found {len(resv)} reservations.")

log_event(ticket_id_3, "TOOL_CALL", tool_name="get_user_profile", details={"user_id": "a4ab87"})
log_event(ticket_id_3, "TOOL_RESULT", tool_name="get_user_profile", outcome="success", details=prof)

upd_3 = update_ticket_status.invoke({"ticket_id": ticket_id_3, "status": "resolved", "issue_type": "account_inquiry"})
log_event(ticket_id_3, "RESOLUTION", agent_name="account_agent", details=upd_3)
print(f"[FINAL STATUS]: {upd_3}")

NameError: name 'classify_ticket' is not defined

## 5. Ticket Scenario 4: Error Handling & Edge Case

- **Query**: Reservation lookup with unknown user ID.
- **Expected Action**: Tool validation catches unknown user -> Structured error returned -> Handled cleanly.

In [1]:
ticket_id_4 = "demo_ticket_edge_004"
user_msg_4 = "Check reservations for non-existent user xyz999."
meta_4 = {"tags": "account", "urgency": "normal", "ticket_id": ticket_id_4}

log_event(ticket_id_4, "CLASSIFICATION", agent_name="supervisor_agent", details=classify_ticket(user_msg_4, meta_4))
resv_err = get_user_reservations.invoke({"user_id": "nonexistent_user_xyz999"})
print(f"[EDGE CASE TOOL RESULT]: {resv_err}")
log_event(ticket_id_4, "TOOL_RESULT", tool_name="get_user_reservations", outcome="error", details=resv_err)

NameError: name 'log_event' is not defined

## 6. Structured Operational Metrics & Log Trace Summary

In [1]:
metrics = get_metrics_summary()
print("=" * 60)
print("            STRUCTURED OPERATIONAL METRICS SUMMARY          ")
print("=" * 60)
print(f"Total Events Recorded     : {metrics['total_events']}")
print(f"Unique Tickets Processed  : {metrics['unique_tickets']}")
print(f"Total Tool Calls Executed : {metrics['total_tool_calls']}")
print(f"Retrieval Success Rate    : {metrics['retrieval_success_rate'] * 100:.1f}%")
print(f"Human Escalation Count   : {metrics['escalation_count']}")
print(f"Tool Usage Breakdown     : {metrics['tool_usage_counts']}")

NameError: name 'get_metrics_summary' is not defined